# [Chapter 19 — HIV, §19.5] Generalized vs Concentrated Epidemics: Mixing Structure Matters

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 19 — HIV
**Considerations developed:** 1 (regime), 7 (mixing balance), 12 (basic vs effective $R$)
**Estimated runtime:** ≤ 20 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_19_hiv/02_HIV_geographic_comparison.ipynb)


## What this notebook does

Compares two HIV epidemic profiles with the *same* aggregate $\mathcal{R}_0$:

- **Generalized epidemic**: heterosexual bipartite transmission across the population (Notebook 01 setting).
- **Concentrated epidemic**: transmission largely within a high-risk subpopulation (e.g., MSM in many high-income settings).

These produce dramatically different prevalence trajectories, intervention sensitivities, and long-run equilibria — even with identical $\mathcal{R}_0$. This is one of the cleanest demonstrations in epidemiology that $\mathcal{R}_0$ alone is insufficient (Consideration 12) and that the *structure* of mixing matters as much as the *magnitude* of contact (Consideration 7).

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_19
from shared.verification import assert_within_tolerance
set_seed_chapter_19()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## Two-group transmission with adjustable assortativity

Single sex, two activity classes. High-risk fraction $\eta = 0.05$, contact-rate ratio $\rho_c = 10$. Assortativity $\epsilon$ sweeps from 0 (proportional) to 1 (fully assortative).

In [ ]:
import math
eta = 0.05         # high-risk fraction
rho_c = 10.0       # high-risk vs low-risk contact ratio
c_low = 1.0
c_high = c_low * rho_c
tau_R = 10.0
beta_per_partnership = 0.05  # tuned so R_0 ~ 2 across configurations

def NGM(epsilon):
    NL, NH = 1-eta, eta
    denom = c_low*NL + c_high*NH
    prop = np.array([[c_low*NL, c_high*NH],[c_low*NL, c_high*NH]])/denom
    M = epsilon*np.eye(2) + (1-epsilon)*prop
    K = np.array([[beta_per_partnership*c_low*M[0,0]*tau_R, beta_per_partnership*c_low*M[0,1]*tau_R],
                  [beta_per_partnership*c_high*M[1,0]*tau_R, beta_per_partnership*c_high*M[1,1]*tau_R]])
    return K, M

for eps in [0.0, 0.5, 1.0]:
    K, _ = NGM(eps)
    R0 = float(np.max(np.real(np.linalg.eigvals(K))))
    print(f'epsilon={eps}: R_0 = {R0:.3f}')


## Long-run prevalence as a function of mixing structure

In [ ]:
def simulate(epsilon, t_end=100):
    K, M = NGM(epsilon)
    NL, NH = 1-eta, eta
    def rhs(y, t):
        SL, IL, SH, IH = y
        Lam_L = beta_per_partnership * c_low * (M[0,0]*IL/NL + M[0,1]*IH/NH)
        Lam_H = beta_per_partnership * c_high * (M[1,0]*IL/NL + M[1,1]*IH/NH)
        return [-Lam_L*SL + IL/tau_R*0, Lam_L*SL - IL/tau_R,  # SI without recovery to S to keep simple endemic
                -Lam_H*SH + IH/tau_R*0, Lam_H*SH - IH/tau_R]
    # Use SIS for tractable steady-state
    def rhs_sis(y, t):
        SL, IL, SH, IH = y
        Lam_L = beta_per_partnership * c_low * (M[0,0]*IL/NL + M[0,1]*IH/NH)
        Lam_H = beta_per_partnership * c_high * (M[1,0]*IL/NL + M[1,1]*IH/NH)
        return [-Lam_L*SL + IL/tau_R, Lam_L*SL - IL/tau_R,
                -Lam_H*SH + IH/tau_R, Lam_H*SH - IH/tau_R]
    y0 = [NL - 1e-4, 1e-4, NH - 1e-4, 1e-4]
    t = np.linspace(0, t_end, 5001)
    sol = odeint(rhs_sis, y0, t)
    return t, sol

fig, ax = plt.subplots(figsize=(10, 4.5))
for eps, color, lbl in [(0.0, BOOK_COLORS['primary'], 'proportional (generalized)'),
                         (0.5, BOOK_COLORS['highlight'], 'mixed'),
                         (1.0, BOOK_COLORS['secondary'], 'fully assortative (concentrated)')]:
    t, sol = simulate(eps)
    SL, IL, SH, IH = sol.T
    overall = (IL + IH) / 1.0
    ax.plot(t, overall*100, color=color, lw=2, label=f'eps={eps}: {lbl}')
ax.set_xlabel('years')
ax.set_ylabel('overall prevalence (%)')
ax.set_title('Same $R_0$ structure, different mixing -> different long-run prevalence')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## High-risk-group prevalence in the concentrated epidemic

In [ ]:
t, sol = simulate(1.0)
SL, IL, SH, IH = sol.T
overall = (IL + IH) / 1.0
high_only = IH / eta
low_only = IL / (1-eta)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t, overall*100, color=BOOK_COLORS['primary'], lw=2, label='overall')
ax.plot(t, high_only*100, color=BOOK_COLORS['secondary'], lw=2, label='within high-risk group')
ax.plot(t, low_only*100, color=BOOK_COLORS['highlight'], lw=2, label='within low-risk group')
ax.set_xlabel('years')
ax.set_ylabel('prevalence (%)')
ax.set_title('Fully assortative mixing: 50%+ within high-risk, ~0% in low-risk')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

high_eq = float(IH[-1]/eta)
overall_eq = float(overall[-1])
print(f'\nFully assortative epidemic at t=100y:')
print(f'  Overall prevalence:        {overall_eq*100:.1f}%')
print(f'  Within high-risk group:    {high_eq*100:.1f}%')
print(f'  Aggregate hides the within-group concentration entirely.')

# Verify: high-risk prevalence vastly exceeds aggregate
assert high_eq > 5*overall_eq, 'high-risk prevalence should be many multiples of aggregate'
print('\nVerified: aggregate prevalence understates within-group risk by 5x or more.')


## What's next

- This phenomenon explains why some interventions (geographically- or behaviorally-targeted prevention) outperform population-wide approaches by orders of magnitude in concentrated epidemics.
- The same algebra applies to vector-borne diseases with spatially-heterogeneous transmission, MDR-TB in healthcare settings, and any scenario with strong within-cluster contact.
- Notebook 03 (perinatal HIV intervention) returns to the bipartite framework and adds vertical-transmission interventions.